In [ ]:
# import os
# import shutil
# import random
# import tensorflow as tf
# import numpy as np

# # ========== CONFIG ==========
# SOURCE_DIR = "potato"
# DATASET_DIR = "dataset"
# IMAGE_SIZE = (256, 256)
# BATCH_SIZE = 32
# EPOCHS = 30
# SPLIT_RATIO = (0.7, 0.15, 0.15)  # Train, Val, Test
# MODEL_OUTPUT = "your_model.h5"
# CLASS_NAMES_OUTPUT = "class_names.txt"

# # ========== GPU CHECK ==========
# device_name = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
# print(f"✅ Using device: {device_name}")

# # ========== STEP 1: SPLIT THE DATA ==========
# def create_split_dirs():
#     for split in ["train", "val", "test"]:
#         split_dir = os.path.join(DATASET_DIR, split)
#         os.makedirs(split_dir, exist_ok=True)

# def split_data():
#     create_split_dirs()
#     for class_name in os.listdir(SOURCE_DIR):
#         class_path = os.path.join(SOURCE_DIR, class_name)
#         if not os.path.isdir(class_path):
#             continue

#         images = os.listdir(class_path)
#         random.shuffle(images)

#         train_end = int(SPLIT_RATIO[0] * len(images))
#         val_end = train_end + int(SPLIT_RATIO[1] * len(images))

#         splits = {
#             "train": images[:train_end],
#             "val": images[train_end:val_end],
#             "test": images[val_end:]
#         }

#         for split, split_imgs in splits.items():
#             target_dir = os.path.join(DATASET_DIR, split, class_name)
#             os.makedirs(target_dir, exist_ok=True)

#             for img_name in split_imgs:
#                 src = os.path.join(class_path, img_name)
#                 dst = os.path.join(target_dir, img_name)
#                 shutil.copy2(src, dst)

# # Only run once
# if not os.path.exists(os.path.join(DATASET_DIR, "train")):
#     print("📁 Splitting data into train/val/test...")
#     split_data()

# # ========== STEP 2: LOAD DATA ==========
# AUTOTUNE = tf.data.AUTOTUNE

# train_ds = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, "train"),
#     label_mode="int",
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )
# val_ds = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, "val"),
#     label_mode="int",
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )
# test_ds = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, "test"),
#     label_mode="int",
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=False
# )

# class_names = train_ds.class_names
# print(f"✅ Classes found: {class_names}")

# # Save class names
# with open(CLASS_NAMES_OUTPUT, "w") as f:
#     for name in class_names:
#         f.write(name + "\n")

# # ========== STEP 3: PREPARE DATA ==========
# def prepare(ds, shuffle=False):
#     ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)
#     if shuffle:
#         ds = ds.shuffle(1000)
#     return ds.prefetch(buffer_size=AUTOTUNE)

# train_ds = prepare(train_ds, shuffle=True)
# val_ds = prepare(val_ds)
# test_ds = prepare(test_ds)

# # ========== STEP 4: DATA AUGMENTATION ==========
# data_augmentation = tf.keras.Sequential([
#     tf.keras.layers.RandomFlip("horizontal"),
#     tf.keras.layers.RandomRotation(0.1),
#     tf.keras.layers.RandomZoom(0.2),
#     tf.keras.layers.RandomContrast(0.1),
# ])




In [ ]:
# # ========== STEP 5: BUILD AND TRAIN MODEL ==========
# with tf.device(device_name):
#     base_model = tf.keras.applications.ResNet50(
#     input_shape=IMAGE_SIZE + (3,),
#     include_top=False,
#     weights='imagenet'
# )

#     base_model.trainable = False

#     model = tf.keras.Sequential([
#         tf.keras.layers.InputLayer(input_shape=IMAGE_SIZE + (3,)),
#         data_augmentation,
#         base_model,
#         tf.keras.layers.GlobalAveragePooling2D(),
#         tf.keras.layers.Dense(128, activation='relu'),
#         tf.keras.layers.Dropout(0.3),
#         tf.keras.layers.Dense(len(class_names), activation='softmax')
#     ])

#     model.compile(
#         optimizer='adam',
#         loss='sparse_categorical_crossentropy',
#         metrics=['accuracy']
#     )
#     callbacks = [
#         tf.keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True),
#         tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5),
#         tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
#         tf.keras.callbacks.TensorBoard(log_dir="logs")
#     ]

#     print("🚀 Starting training...")
#     history = model.fit(
#         train_ds,
#         validation_data=val_ds,
#         epochs=EPOCHS,
#         callbacks=callbacks
#     )

#     print("✅ Training complete. Evaluating...")
#     loss, acc = model.evaluate(test_ds)
#     print(f"✅ Final Test Accuracy: {acc:.4f}")

#     model.save(MODEL_OUTPUT)
#     print(f"✅ Model saved to: {MODEL_OUTPUT}")


In [ ]:
!pip install scikit-learn


In [ ]:
import os
import shutil
import random
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications.resnet50 import preprocess_input


# ========== CONFIG ==========
SOURCE_DIR = "potato"
DATASET_DIR = "dataset"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 20
MODEL_OUTPUT = "your_model.h5"
CLASS_NAMES_OUTPUT = "class_names.txt"

device_name = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print(f"✅ Using device: {device_name}")

# ========== STEP 1: SPLIT DATA ==========
def create_split_dirs():
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(DATASET_DIR, split), exist_ok=True)

def split_data():
    create_split_dirs()
    for class_name in os.listdir(SOURCE_DIR):
        class_path = os.path.join(SOURCE_DIR, class_name)
        if not os.path.isdir(class_path):
            continue

        images = os.listdir(class_path)
        random.shuffle(images)

        train_end = int(0.7 * len(images))
        val_end = train_end + int(0.15 * len(images))

        splits = {
            "train": images[:train_end],
            "val": images[train_end:val_end],
            "test": images[val_end:]
        }

        for split, split_imgs in splits.items():
            target_dir = os.path.join(DATASET_DIR, split, class_name)
            os.makedirs(target_dir, exist_ok=True)
            for img in split_imgs:
                shutil.copy2(os.path.join(class_path, img), os.path.join(target_dir, img))

if not os.path.exists(os.path.join(DATASET_DIR, "train")):
    print("📁 Splitting data into train/val/test...")
    split_data()

# ========== STEP 2: LOAD DATA ==========
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "val"),
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "test"),
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
with open(CLASS_NAMES_OUTPUT, "w") as f:
    for name in class_names:
        f.write(name + "\n")
print(f"✅ Classes found: {class_names}")

# ========== STEP 3: IMAGE AUGMENTATION + NORMALIZATION ==========
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.3),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(factor=0.2),
])



def preprocess(x, y):
    x = preprocess_input(x)  # applies proper preprocessing
    return x, y

def prepare(ds, shuffle=False):
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000)
    return ds.prefetch(buffer_size=AUTOTUNE)

train_ds = prepare(train_ds, shuffle=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

# ========== STEP 4: COMPUTE CLASS WEIGHTS ==========
all_labels = np.concatenate([y.numpy() for x, y in train_ds])
class_weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# ========== STEP 5: BUILD MODEL ==========
with tf.device(device_name):
    base_model = tf.keras.applications.ResNet50(
        input_shape=IMAGE_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
    x = data_augmentation(inputs)
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

✅ Using device: /GPU:0
Found 3990 files belonging to 3 classes.
Found 853 files belonging to 3 classes.
Found 859 files belonging to 3 classes.
✅ Classes found: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
Class weights: {0: 0.9801031687546058, 1: 0.9801031687546058, 2: 1.042319749216301}


In [13]:
# ========== STEP 6: TRAINING PHASE 1 (Feature Extraction) ==========
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
]

print("🚀 Starting phase 1: Feature Extraction...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

# ========== STEP 7: FINE-TUNING PHASE ==========
print("🔓 Unfreezing top layers for fine-tuning...")
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


print("🚀 Starting phase 2: Fine-tuning...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

# ========== EVALUATE ==========
print("✅ Training complete. Evaluating...")
loss, acc = model.evaluate(test_ds)
print(f"✅ Final Test Accuracy: {acc:.4f}")

model.save(MODEL_OUTPUT)
print(f"✅ Model saved to: {MODEL_OUTPUT}")

🚀 Starting phase 1: Feature Extraction...
Epoch 1/10
125/125 [==============================] - 73s 511ms/step - loss: 0.4068 - accuracy: 0.8474 - val_loss: 0.3099 - val_accuracy: 0.8593 - lr: 0.0010
Epoch 2/10
125/125 [==============================] - 67s 516ms/step - loss: 0.2606 - accuracy: 0.9048 - val_loss: 0.1922 - val_accuracy: 0.9179 - lr: 0.0010
Epoch 3/10
125/125 [==============================] - 67s 513ms/step - loss: 0.2421 - accuracy: 0.9120 - val_loss: 0.3352 - val_accuracy: 0.8687 - lr: 0.0010
Epoch 4/10
125/125 [==============================] - 65s 502ms/step - loss: 0.2369 - accuracy: 0.9093 - val_loss: 0.3066 - val_accuracy: 0.8734 - lr: 0.0010
Epoch 5/10
125/125 [==============================] - 66s 504ms/step - loss: 0.1968 - accuracy: 0.9271 - val_loss: 0.2146 - val_accuracy: 0.9074 - lr: 0.0010
Epoch 6/10
125/125 [==============================] - 66s 506ms/step - loss: 0.1975 - accuracy: 0.9248 - val_loss: 0.3848 - val_accuracy: 0.8441 - lr: 5.0000e-04
Epoch 

In [ ]:
# import matplotlib.pyplot as plt

# def plot_history(histories):
#     acc = []
#     val_acc = []
#     loss = []
#     val_loss = []

#     for h in histories:
#         acc += h.history["accuracy"]
#         val_acc += h.history["val_accuracy"]
#         loss += h.history["loss"]
#         val_loss += h.history["val_loss"]

#     plt.figure(figsize=(14,5))
#     plt.subplot(1,2,1)
#     plt.plot(acc, label='Train Acc')
#     plt.plot(val_acc, label='Val Acc')
#     plt.legend(); plt.grid(); plt.title("Accuracy")

#     plt.subplot(1,2,2)
#     plt.plot(loss, label='Train Loss')
#     plt.plot(val_loss, label='Val Loss')
#     plt.legend(); plt.grid(); plt.title("Loss")

# plot_history([history_initial, history_fine])


NameError: name 'history_initial' is not defined

In [ ]:
# import os
# import shutil
# import random
# import tensorflow as tf
# import numpy as np
# from sklearn.utils.class_weight import compute_class_weight
# from tensorflow.keras.applications.resnet50 import preprocess_input

# # ========== CONFIG ==========
# SOURCE_DIR = "potato"
# DATASET_DIR = "dataset"
# IMAGE_SIZE = (224, 224)  # ✅ Changed to ResNet50 default
# BATCH_SIZE = 32
# INITIAL_EPOCHS = 10
# FINE_TUNE_EPOCHS = 50
# MODEL_OUTPUT = "your_model.h5"
# CLASS_NAMES_OUTPUT = "class_names.txt"

# device_name = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
# print(f"✅ Using device: {device_name}")

# # ========== STEP 1: SPLIT DATA ==========
# def create_split_dirs():
#     for split in ["train", "val", "test"]:
#         os.makedirs(os.path.join(DATASET_DIR, split), exist_ok=True)

# def split_data():
#     create_split_dirs()
#     for class_name in os.listdir(SOURCE_DIR):
#         class_path = os.path.join(SOURCE_DIR, class_name)
#         if not os.path.isdir(class_path):
#             continue
#         images = os.listdir(class_path)
#         random.shuffle(images)
#         train_end = int(0.7 * len(images))
#         val_end = train_end + int(0.15 * len(images))
#         splits = {
#             "train": images[:train_end],
#             "val": images[train_end:val_end],
#             "test": images[val_end:]
#         }
#         for split, split_imgs in splits.items():
#             target_dir = os.path.join(DATASET_DIR, split, class_name)
#             os.makedirs(target_dir, exist_ok=True)
#             for img in split_imgs:
#                 shutil.copy2(os.path.join(class_path, img), os.path.join(target_dir, img))

# if not os.path.exists(os.path.join(DATASET_DIR, "train")):
#     print("📁 Splitting data into train/val/test...")
#     split_data()

# # ========== STEP 2: LOAD DATA ==========
# AUTOTUNE = tf.data.AUTOTUNE

# train_ds = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, "train"),
#     label_mode="int",
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )
# val_ds = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, "val"),
#     label_mode="int",
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )
# test_ds = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATASET_DIR, "test"),
#     label_mode="int",
#     image_size=IMAGE_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=False
# )

# class_names = train_ds.class_names
# print(f"✅ Classes found: {class_names}")

# # ✅ Save class names
# with open(CLASS_NAMES_OUTPUT, "w") as f:
#     for name in class_names:
#         f.write(name + "\n")

# # ========== STEP 3: AUGMENT + PREPROCESS ==========
# data_augmentation = tf.keras.Sequential([
#     tf.keras.layers.RandomFlip("horizontal_and_vertical"),
#     tf.keras.layers.RandomRotation(0.2),
#     tf.keras.layers.RandomZoom(0.3),
#     tf.keras.layers.RandomContrast(0.2),
# ])

# def preprocess(x, y):
#     x = preprocess_input(tf.cast(x, tf.float32))
#     return x, y

# def prepare(ds, shuffle=False):
#     ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
#     if shuffle:
#         ds = ds.shuffle(1000)
#     return ds.prefetch(AUTOTUNE)

# train_ds = prepare(train_ds, shuffle=True)
# val_ds = prepare(val_ds)
# test_ds = prepare(test_ds)

# # ========== STEP 4: COMPUTE CLASS WEIGHTS ==========
# # Extract all labels from train_ds for class weight calculation
# train_labels = []
# for _, labels in train_ds.unbatch():
#     train_labels.append(labels.numpy())

# all_labels = np.array(train_labels)
# class_weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
# class_weight_dict = dict(enumerate(class_weights))

# # ========== STEP 5: BUILD MODEL ==========
# with tf.device(device_name):
#     base_model = tf.keras.applications.ResNet50(
#         input_shape=IMAGE_SIZE + (3,),
#         include_top=False,
#         weights="imagenet"
#     )
#     base_model.trainable = False  # Initial training only top layers

#     inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
#     x = data_augmentation(inputs)
#     x = base_model(x, training=False)
#     x = tf.keras.layers.GlobalAveragePooling2D()(x)
#     x = tf.keras.layers.Dense(256, activation="relu")(x)
#     x = tf.keras.layers.BatchNormalization()(x)
#     x = tf.keras.layers.Dropout(0.4)(x)
#     outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

#     model = tf.keras.Model(inputs, outputs)

#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(),
#         loss="sparse_categorical_crossentropy",
#         metrics=["accuracy"]
#     )

# # ========== STEP 6: TRAIN FEATURE EXTRACTOR ==========
# callbacks = [
#     tf.keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True),
#     tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5),
#     tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#     tf.keras.callbacks.TensorBoard(log_dir="logs"),
# ]

# print("🚀 Training feature extractor...")
# history_initial = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=INITIAL_EPOCHS,
#     class_weight=class_weight_dict,
#     callbacks=callbacks
# )

# # ========== STEP 7: FINE-TUNE MODEL ==========
# base_model.trainable = True
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-5),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# print("🔧 Fine-tuning entire model...")
# history_fine = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=FINE_TUNE_EPOCHS,
#     class_weight=class_weight_dict,
#     callbacks=callbacks
# )

# # ========== STEP 8: EVALUATE ==========
# print("✅ Evaluating on test set...")
# loss, acc = model.evaluate(test_ds)
# print(f"🎯 Final Test Accuracy: {acc:.4f}")

# # ========== STEP 9: SAVE ==========
# model.save(MODEL_OUTPUT)
# print(f"📦 Model saved to: {MODEL_OUTPUT}")


✅ Using device: /GPU:0
Found 3990 files belonging to 3 classes.
Found 853 files belonging to 3 classes.
Found 859 files belonging to 3 classes.
✅ Classes found: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
🚀 Training feature extractor...
Epoch 1/10
125/125 [==============================] - 65s 457ms/step - loss: 0.2430 - accuracy: 0.9110 - val_loss: 0.3324 - val_accuracy: 0.8675 - lr: 0.0010
Epoch 2/10
 43/125 [=========>....................] - ETA: 33s - loss: 0.1779 - accuracy: 0.9346

KeyboardInterrupt: 